# no-relu-on-final-layer — faded example 1: complete the fixed classifier forward

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `no-relu-on-final-layer`. Running the beacon reports progress on the `CNN: No-ReLU on final layer` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: No-ReLU on final layer` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`no-relu-on-final-layer`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "no-relu-on-final-layer"
DD_SUBTOPIC = "CNN: No-ReLU on final layer"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The fix for a stray final ReLU is a forward pass that applies the activation only between layers and returns the last layer's raw output. The final return must be the bare `fc2(h)`.

## Faded exercise 1

Complete the `forward` of `FixedClassifier` so it applies ReLU only between `fc1` and `fc2`, returning raw logits. Fill in the final return.

**Fill in:** the forward return producing raw logits from fc2 with no final activation

In [ ]:
import torch as t
import torch.nn as nn
import torch.nn.functional as F

t.manual_seed(3)

class BrokenClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(8, 16)
        self.fc2 = nn.Linear(16, 4)
    def forward(self, x):
        return F.relu(self.fc2(F.relu(self.fc1(x))))

class FixedClassifier(nn.Module):
    def __init__(self, fc1, fc2):
        super().__init__()
        self.fc1 = fc1
        self.fc2 = fc2
    def forward(self, x):
        h = F.relu(self.fc1(x))
        return self.fc2(h)

b = BrokenClassifier()
fixed = FixedClassifier(b.fc1, b.fc2)
print((fixed(t.randn(32, 8)) < 0).any().item())


def _test():
    t.manual_seed(33)
    b = BrokenClassifier()
    fixed = FixedClassifier(b.fc1, b.fc2)
    x = t.randn(128, 8)
    fixed.eval()
    with t.no_grad():
        y = fixed(x)
    # fixed model must produce some negative logits
    assert (y < 0).any().item()
    # and it must reuse the broken weights (same object)
    assert fixed.fc1 is b.fc1 and fixed.fc2 is b.fc2
    # broken model essentially never negative on its final output
    with t.no_grad():
        yb = b(x)
    assert (yb < 0).float().mean().item() < 1e-3


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import torch.nn as nn
import torch.nn.functional as F

t.manual_seed(3)

class BrokenClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(8, 16)
        self.fc2 = nn.Linear(16, 4)
    def forward(self, x):
        return F.relu(self.fc2(F.relu(self.fc1(x))))

class FixedClassifier(nn.Module):
    def __init__(self, fc1, fc2):
        super().__init__()
        self.fc1 = fc1
        self.fc2 = fc2
    def forward(self, x):
        h = F.relu(self.fc1(x))
        return self.fc2(h)

b = BrokenClassifier()
fixed = FixedClassifier(b.fc1, b.fc2)
print((fixed(t.randn(32, 8)) < 0).any().item())
```
</details>